# Lecture 05 - Solutions to Part B (Python Applications)

This notebook gives worked Python solutions for Exercises 15-22. The code is intentionally written with simple building blocks: `pandas`, `numpy`, `matplotlib`, and a small normal-CDF helper based on `math.erf`, so the normal calculations are transparent.

In [ ]:
from pathlib import Path
from math import erf, sqrt, exp, pi

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:.6f}".format)
plt.style.use("seaborn-v0_8-whitegrid")

DATA = Path("data and notebooks")

def normal_cdf(x, mean=0.0, sd=1.0):
    """Return P(X <= x) for X following a normal distribution."""
    z = (x - mean) / sd
    return 0.5 * (1 + erf(z / sqrt(2)))

def normal_pdf(x, mean=0.0, sd=1.0):
    """Return normal density values for scalar or array x."""
    x = np.asarray(x)
    z = (x - mean) / sd
    return (1 / (sd * sqrt(2 * pi))) * np.exp(-0.5 * z**2)

def interval_probability_from_cdf(a, b, mean=0.0, sd=1.0):
    return normal_cdf(b, mean, sd) - normal_cdf(a, mean, sd)


---

## Exercise 15 - Empirical Distribution of Monthly Returns

In [ ]:
monthly = pd.read_csv(DATA / "Lecture 05 monthly returns.csv")
monthly["date"] = pd.to_datetime(monthly["date"])
monthly_return = pd.to_numeric(monthly["monthly_return"], errors="coerce").dropna()

monthly_summary = monthly_return.describe(percentiles=[0.25, 0.5, 0.75])
monthly_summary

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(monthly_return, bins=18, edgecolor="white", alpha=0.8)
plt.axvline(monthly_return.mean(), color="black", linestyle="--", label="Sample mean")
plt.title("Empirical Distribution of Monthly Returns")
plt.xlabel("Monthly return")
plt.ylabel("Number of months")
plt.legend()
plt.show()

In [ ]:
between_1_and_2 = monthly_return.between(0.01, 0.02, inclusive="neither").mean()
count_between_1_and_2 = monthly_return.between(0.01, 0.02, inclusive="neither").sum()

pd.Series({
    "count_between_1pct_and_2pct": count_between_1_and_2,
    "empirical_probability": between_1_and_2,
})

**Interpretation.** The histogram and summary describe the empirical distribution of the observed monthly returns. The empirical probability is a relative frequency in this sample, not a model probability for all future months.

---

## Exercise 16 - Waiting Times and Skewness

In [ ]:
waiting = pd.read_csv(DATA / "Lecture 05 shop waiting times.csv")
wait = pd.to_numeric(waiting["waiting_time_minutes"], errors="coerce").dropna()

wait_summary = wait.describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
wait_summary

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(wait, bins=25, edgecolor="white", alpha=0.8)
plt.axvline(wait.mean(), color="black", linestyle="--", label="Mean")
plt.axvline(wait.median(), color="tab:orange", linestyle="--", label="Median")
plt.title("Shop Waiting Times")
plt.xlabel("Waiting time (minutes)")
plt.ylabel("Number of customers")
plt.legend()
plt.show()

In [ ]:
wait_between_1_and_2 = wait.between(1, 2, inclusive="neither").mean()
wait_comparison = pd.Series({
    "mean": wait.mean(),
    "median": wait.median(),
    "mean_minus_median": wait.mean() - wait.median(),
    "empirical_probability_1_to_2_minutes": wait_between_1_and_2,
})
wait_comparison

**Interpretation.** Waiting times are right-skewed: most observations are short, but a few long waits pull the mean above the median. A symmetric normal model would be questionable because it can place probability on negative waits and does not naturally represent the long right tail.

---

## Exercise 17 - Uniform Distribution in Python

In [ ]:
def uniform_interval_probability(a, b, low=10, high=30):
    """Compute P(a < X < b) for X uniformly distributed on [low, high]."""
    left = max(a, low)
    right = min(b, high)
    covered_length = max(0, right - left)
    return covered_length / (high - low)

uniform_results = pd.Series({
    "P(12 < X < 18)": uniform_interval_probability(12, 18),
    "P(25 < X < 30)": uniform_interval_probability(25, 30),
    "P(5 < X < 12)": uniform_interval_probability(5, 12),
})
uniform_results

**Interpretation.** The function first clips the requested interval to the support `[10, 30]`. Only the overlap with the support receives probability. For example, the interval `(5, 12)` contributes only the length from 10 to 12.

---

## Exercise 18 - Normal Demand Probability

In [ ]:
mu = 100
sigma = 15

prob_direct = interval_probability_from_cdf(90, 120, mean=mu, sd=sigma)
z_90 = (90 - mu) / sigma
z_120 = (120 - mu) / sigma
prob_standardized = normal_cdf(z_120) - normal_cdf(z_90)

pd.Series({
    "z_90": z_90,
    "z_120": z_120,
    "direct_probability": prob_direct,
    "standardized_probability": prob_standardized,
    "difference": prob_direct - prob_standardized,
})

**Interpretation.** The direct and standardized calculations agree because they describe the same area under the normal curve on two different horizontal scales. Under the model, demand falls between 90 and 120 units with probability about 65.6%.

---

## Exercise 19 - Fitting a Normal Model to Netflix Returns

In [ ]:
nflx = pd.read_csv(DATA / "Lecture 05 nflx yahoo daily log returns 2015 2026.csv")
nflx["date"] = pd.to_datetime(nflx["date"])
returns = pd.to_numeric(nflx["log_return"], errors="coerce").dropna()

mu_hat = returns.mean()
sigma_hat = returns.std(ddof=1)

fit_summary = pd.Series({
    "observations": returns.size,
    "mu_hat": mu_hat,
    "sigma_hat": sigma_hat,
    "min": returns.min(),
    "max": returns.max(),
})
fit_summary

In [ ]:
x_grid = np.linspace(returns.min(), returns.max(), 600)

plt.figure(figsize=(9, 4.5))
plt.hist(returns, bins=80, density=True, alpha=0.65, edgecolor="white", label="Empirical returns")
plt.plot(x_grid, normal_pdf(x_grid, mu_hat, sigma_hat), color="tab:orange", lw=2, label="Fitted normal density")
plt.title("Netflix Daily Log Returns: Empirical Histogram and Fitted Normal Model")
plt.xlabel("Daily log return")
plt.ylabel("Density")
plt.legend()
plt.show()

**Interpretation.** The fitted normal model uses the sample mean as `mu_hat` and the sample standard deviation as `sigma_hat`. It is a benchmark model for the distribution of returns, not proof that returns are truly normal.

---

## Exercise 20 - Empirical Versus Normal Probabilities

In [ ]:
def empirical_between(series, a, b):
    return ((series > a) & (series < b)).mean()

comparison = pd.DataFrame([
    {
        "event": "-0.02 < X < 0.02",
        "empirical": empirical_between(returns, -0.02, 0.02),
        "normal_model": interval_probability_from_cdf(-0.02, 0.02, mu_hat, sigma_hat),
    },
    {
        "event": "-0.05 < X < -0.02",
        "empirical": empirical_between(returns, -0.05, -0.02),
        "normal_model": interval_probability_from_cdf(-0.05, -0.02, mu_hat, sigma_hat),
    },
    {
        "event": "X < -0.03",
        "empirical": (returns < -0.03).mean(),
        "normal_model": normal_cdf(-0.03, mu_hat, sigma_hat),
    },
    {
        "event": "X > 0.04",
        "empirical": (returns > 0.04).mean(),
        "normal_model": 1 - normal_cdf(0.04, mu_hat, sigma_hat),
    },
])
comparison["difference_empirical_minus_model"] = comparison["empirical"] - comparison["normal_model"]
comparison

**Interpretation.** The model and the data do not agree equally well in every region. In this dataset, the empirical center is more concentrated than the fitted normal model suggests, while some tail comparisons also differ. These are model diagnostics, not just calculations.

---

## Exercise 21 - Tail Risk Under the Normal Model

In [ ]:
left_tail_threshold = mu_hat - 3 * sigma_hat
empirical_left_tail = (returns < left_tail_threshold).mean()
normal_left_tail = normal_cdf(left_tail_threshold, mu_hat, sigma_hat)

tail_summary = pd.Series({
    "mu_hat": mu_hat,
    "sigma_hat": sigma_hat,
    "left_tail_threshold_mu_minus_3sigma": left_tail_threshold,
    "empirical_probability": empirical_left_tail,
    "normal_model_probability": normal_left_tail,
    "empirical_to_model_ratio": empirical_left_tail / normal_left_tail,
})
tail_summary

**Interpretation.** The empirical probability of being more than three fitted standard deviations below the mean is much larger than the normal benchmark. This matters because risk analysis often focuses precisely on rare downside events.

---

## Exercise 22 - Careful Probability Memo

In [ ]:
memo = f"""
Using Netflix daily log returns, the fitted normal model has mean {mu_hat:.4%} and standard deviation {sigma_hat:.4%}.
For the event -2% < X < 2%, the empirical frequency is {comparison.loc[0, 'empirical']:.1%}, while the fitted normal model gives {comparison.loc[0, 'normal_model']:.1%}.
The empirical frequency describes what happened in the observed sample; the normal-model probability describes what the fitted model implies.
Because financial returns can have heavier tails than a normal distribution, especially on the downside, the normal model should be used as a benchmark and checked before being used for risk analysis.
"""
print(memo)